# AC 209B Final Project — Milestone 3

## Vanilla BERT Baseline: Headline → Next-Day Realized Volatility

This notebook establishes the first text-only model in our pipeline. We fine-tune `bert-base-uncased` to take a stock's news headlines for day **t** and predict the next-day Garman–Klass realized log-volatility (`log_gk_volatility`) for day **t+1**.

We compare three training regimes against simple naive baselines:

1. **Frozen BERT (linear probe)** — only the regression head is trainable; measures how much signal is already in off-the-shelf BERT embeddings.
2. **Partial fine-tune** — last 2 transformer layers + head; the cheap middle ground.
3. **Full fine-tune** — all parameters trainable; the standard approach.

All three variants share the chronological train/val/test split established in `AC209B_Final_Project_Milestone3.ipynb`, with the test set split into a Normal sub-period (Jan–Feb 2020) and a COVID sub-period (Mar–Jun 2020) to surface distribution-shift behavior.

---

# Part 0: Setup

## 0.1 Imports

We import PyTorch for model training, the Hugging Face `transformers` library for BERT and its tokenizer, scikit-learn for evaluation metrics, and the standard scientific Python stack. We also pin a consistent matplotlib style to match the main MS3 notebook.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use("seaborn-v0_8-whitegrid")

## 0.2 Device & Reproducibility

In [ ]:
# Reproducibility
SEED = 1090


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic runs are slower; enable if you need exact reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# Device selection
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("torch:", torch.__version__)
print("device:", DEVICE)

## 0.3 Mount Drive and Set Paths

This notebook runs locally and on Colab. On Colab we mount Google Drive and point at `MyDrive/cs209b/MS3`; locally, `MS3_DIR` is the current directory. `DATA_DIR` is where all checkpoints, history pickles, and the preprocessed parquet live.

In [ ]:
try:
    import google.colab
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    MS3_DIR = Path("/content/drive/MyDrive/cs209b/MS3")
except ImportError:
    MS3_DIR = Path(".")

DATA_DIR = MS3_DIR / "data"

print(f"MS3_DIR  = {MS3_DIR.resolve()}")
print(f"DATA_DIR = {DATA_DIR.resolve()}")
assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"

## 0.4 Load Preprocessed Data

We load `expanded_equities_merged_day_gk_hist.parquet`, the cleaned dataset from `merge_gk_weekly_adf.ipynb`. Each row is one **stock-day pair**: all headlines for that ticker on that date concatenated with `"; "`, plus 50 historical OHLCV features (which we ignore in this baseline), `sector`, `industry`, `gk_volatility`, and the regression target `log_gk_volatility`. The `t_plus_1_*` columns have already been dropped to prevent leakage.

In [ ]:
DATA_PATH = DATA_DIR / "expanded_equities_merged_day_gk_hist.parquet"

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(f"{'shape':<8} = {df.shape}")
print(f"{'columns':<8} = {list(df.columns[:8])} ...")
print(f"date range: {df['date'].min().date()} → {df['date'].max().date()}")
df[["date", "stock", "title", "log_gk_volatility"]].head(3)

## 0.5 Reproduce Chronological Splits

We use the exact same chronological boundaries outline in main MS3 notebook to keep all model variants comparable with the EDA already done by the team. Train covers 2010-07-24 through 2017-12-31, validation covers 2018–2019, and test is split at March 1, 2020 to isolate pre-COVID generalization (`test_normal`) from the COVID regime shift (`test_covid`). For this baseline we keep only `(stock, date, title, log_gk_volatility)` — text and target.

In [ ]:
KEEP = ["stock", "date", "title", "log_gk_volatility"]

train = df.loc[df["date"] < "2018-01-01", KEEP].reset_index(drop=True)
val = df.loc[
    (df["date"] >= "2018-01-01") & (df["date"] < "2020-01-01"), KEEP
].reset_index(drop=True)
test_normal = df.loc[
    (df["date"] >= "2020-01-01") & (df["date"] < "2020-03-01"), KEEP
].reset_index(drop=True)
test_covid = df.loc[df["date"] >= "2020-03-01", KEEP].reset_index(drop=True)

for name, split in [
    ("Train", train),
    ("Val", val),
    ("Test (Normal)", test_normal),
    ("Test (COVID)", test_covid),
]:
    print(
        f"{name:14}: {len(split):>7,} rows  ({split['date'].min().date()} → {split['date'].max().date()})  "
        f"mean={split['log_gk_volatility'].mean():.3f}  std={split['log_gk_volatility'].std():.3f}"
    )

# Part 1: Tokenization & Datasets

## 1.1 Token-Length Sanity Check

Before fixing a `MAX_LEN`, we check how long our concatenated headlines actually are in BERT's WordPiece tokenizer. Many stock-days have multiple headlines glued together with `"; "`, so the distribution is right-skewed — a single `MAX_LEN` that's too short silently truncates the tail, while one that's too long wastes GPU memory on padding. We tokenize a 20k-row sample of the training set and read off the 50th, 95th, and 99th percentiles. A good `MAX_LEN` covers ≥ 95% of titles without truncation but stays within the BERT limit of 512.

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Inspect longest headlines in the training set
lens = (
    train["title"]
    .fillna("")
    .astype(str)
    .apply(lambda t: len(tokenizer.encode(t, add_special_tokens=True)))
    .to_numpy()
)
print(f"{'n':<8} = {len(lens):,}")
print(f"{'mean':<8} = {lens.mean():.1f}")
print(f"{'p50':<8} = {np.percentile(lens, 50):.0f}")
print(f"{'p95':<8} = {np.percentile(lens, 95):.0f}")
print(f"{'p99':<8} = {np.percentile(lens, 99):.0f}")
print(f"{'max':<8} = {lens.max()}")
print(f"truncated at {MAX_LEN}: {(lens > MAX_LEN).mean() * 100:.2f}% of train rows")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.clip(lens, 0, 400), bins=80, edgecolor="black", alpha=0.8)
ax.axvline(MAX_LEN, color="orange", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
ax.set_title(
    "WordPiece token length of concatenated headlines (full train, clipped at 400)"
)
ax.set_xlabel("tokens per stock-day")
ax.set_ylabel("frequency")
ax.legend()
plt.tight_layout()
plt.show()


### Inspect some longest headlines in the full training set

In [ ]:
lens_full = (
    train["title"]
    .fillna("")
    .astype(str)
    .apply(lambda t: len(tokenizer.encode(t, add_special_tokens=True)))
)

longest = train.assign(tok_len=lens_full).nlargest(10, "tok_len")[
    ["date", "stock", "tok_len", "title"]
]
count = 0

for _, row in longest.iterrows():
    print(f"--- {row['date'].date()}  {row['stock']}  ({row['tok_len']} tokens) ---")
    print(row["title"])
    print()
    count += 1
    if count >= 5:
        break

## 1.2 Dataset & DataLoader

We wrap each split in `torch.utils.data.Dataset` that yields already-tokenized tensors. Titles longer than `MAX_LEN` are truncated; shorter ones are padded. The target is cast to `float32` for the MSE loss. We build DataLoaders for all four splits; the training loader shuffles while the evaluation loaders don't.

In [ ]:
BATCH_SIZE = 256


class HeadlineDataset(Dataset):
    # encoder at init
    def __init__(self, titles, targets, tokenizer, max_len: int = MAX_LEN):
        enc = tokenizer(
            list(titles.fillna("").astype(str)),
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.targets = torch.tensor(targets.values, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.targets)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "target": self.targets[idx],
        }


def make_loader(
    split_df: pd.DataFrame, shuffle: bool = False, batch_size: int = BATCH_SIZE
) -> DataLoader:
    ds = HeadlineDataset(split_df["title"], split_df["log_gk_volatility"], tokenizer)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True
    )


train_loader = make_loader(train, shuffle=True)
val_loader = make_loader(val)
test_normal_loader = make_loader(test_normal)
test_covid_loader = make_loader(test_covid)

print(f"{'train':<14} batches = {len(train_loader):,}")
print(f"{'val':<14} batches = {len(val_loader):,}")
print(f"{'test_normal':<14} batches = {len(test_normal_loader):,}")
print(f"{'test_covid':<14} batches = {len(test_covid_loader):,}")

# Part 2: Model Definition

## 2.1 BertRegressor

This method wraps a pretrained BERT encoder with a single linear regression head applied to the `[CLS]` token's hidden state.\\
`[CLS]` is BERT's designated "whole sequence" summary position, during pre-training it aggregates information from the entire input.
We add a single dropout layer before the head for regularization but keep the head deliberately simple: a `Linear(768 → 1)` maps the `[CLS]` embedding to a scalar `log_gk_volatility` prediction. Anything fancier (MLP head, mean-pooling, attention pooling) is a follow-up — the baseline should be as vanilla as possible.

In [ ]:
class BertRegressor(nn.Module):
    def __init__(self, model_name: str = MODEL_NAME, dropout: float = 0.1):
        super().__init__()
        # from_pretrained downloads the model if not cached locally, then loads it
        self.bert = AutoModel.from_pretrained(model_name).float()
        # dropout before entering the FFNN
        self.dropout = nn.Dropout(dropout)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        return self.regressor(self.dropout(cls)).squeeze(-1)


# clean up GPU memory after model definition
def gpu_cleanup(obj: list[object]):
    for o in obj:
        del o
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None


# test a forward pass
_model = BertRegressor().to(DEVICE)
_batch = next(iter(val_loader))
with torch.no_grad():
    _pred = _model(_batch["input_ids"].to(DEVICE), _batch["attention_mask"].to(DEVICE))
print(f"{'input':<12} = {tuple(_batch['input_ids'].shape)}")
print(f"{'output':<12} = {tuple(_pred.shape)}  dtype={_pred.dtype}")
print(f"{'params':<12} = {sum(p.numel() for p in _model.parameters()):,}")
gpu_cleanup([_model, _batch, _pred])

In [ ]:
# print model architecture
_model = BertRegressor().to(DEVICE)
print(_model)

## 2.2 Freeze Helper

To support the three training variants (frozen, partial, full), we need a utility that freezes all BERT parameters except the last `n_unfrozen` transformer layers. BERT-base has 12 encoder layers; freezing the bottom 10 leaves only the top 2 adaptable — these are the layers that carry the most task-specific representations after fine-tuning, while the bottom layers carry mostly general syntactic/lexical features that don't need to change for our task. The regression head is always trainable. We also print a summary of what's frozen versus trainable so we can double-check before training.

In [ ]:
def freeze_bert_layers(model: BertRegressor, n_unfrozen: int = 0) -> None:
    """Freeze BERT params, then unfreeze the top `n_unfrozen` transformer layers.

    n_unfrozen = 0   → frozen BERT (linear probe); only the head trains
    n_unfrozen = 2   → partial fine-tune: top 2 layers + head
    n_unfrozen = 12  → full fine-tune (all layers trainable)
    """
    # Freeze everything in BERT (embeddings + all encoder layers + pooler)
    for p in model.bert.parameters():
        p.requires_grad = False

    # Unfreeze the top `n_unfrozen` encoder layers
    if n_unfrozen > 0:
        for layer in model.bert.encoder.layer[-n_unfrozen:]:
            for p in layer.parameters():
                p.requires_grad = True


def param_summary(model: nn.Module) -> None:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total = trainable + frozen
    print(f"{'trainable':<12} = {trainable:>12,}  ({trainable / total * 100:5.2f}%)")
    print(f"{'frozen':<12} = {frozen:>12,}  ({frozen / total * 100:5.2f}%)")
    print(f"{'total':<12} = {total:>12,}")


# Quick demo on a throwaway instance
_m = BertRegressor()
print("--- all layers trainable (full fine-tune) ---")
param_summary(_m)
print("\n--- linear probe (n_unfrozen = 0) ---")
freeze_bert_layers(_m, n_unfrozen=0)
param_summary(_m)
print("\n--- partial fine-tune (n_unfrozen = 2) ---")
freeze_bert_layers(_m, n_unfrozen=2)
param_summary(_m)
del _m

# Part 3: Naive Baselines

Before training anything we establish two trivial predictors that BERT must beat. Their metrics form the floor — a "model" that cannot outperform the mean of the training target is not a model at all.

## 3.1 Mean Predictor

The simplest possible baseline: ignore the headline entirely and always predict the training set's mean `log_gk_volatility`. This predictor's R² on any split is exactly zero if the split shares the train's mean, and negative if the split's mean differs. It is the lower bound of "sensible" regression performance.

In [ ]:
def regression_metrics(preds: np.ndarray, targets: np.ndarray) -> dict:
    return {
        "rmse": float(np.sqrt(mean_squared_error(targets, preds))),
        "mae": float(mean_absolute_error(targets, preds)),
        "r2": float(r2_score(targets, preds)),
    }


# All baseline + model results live here; Part 7 compiles them into a table
results: dict[tuple[str, str], dict] = {}

mean_pred = train["log_gk_volatility"].mean()
print(f"train mean = {mean_pred:.4f}\n")

for name, split in [
    ("val", val),
    ("test_normal", test_normal),
    ("test_covid", test_covid),
]:
    preds = np.full(len(split), mean_pred)
    results[("Mean", name)] = regression_metrics(
        preds, split["log_gk_volatility"].values
    )
    m = results[("Mean", name)]
    print(f"  {name:<12}: rmse={m['rmse']:.4f}  mae={m['mae']:.4f}  r2={m['r2']:+.4f}")

## 3.2 Per-Stock Mean Predictor

A smarter baseline that ignores the headline but uses the ticker: predict each stock's *training-set* mean `log_gk_volatility`. This captures the fact that some stocks are structurally more volatile than others (small caps vs utilities). Tickers that appear in val/test but never in train fall back to the global mean. If BERT on headline text alone cannot beat this, we should prefer the baseline — the model isn't learning anything the ticker symbol doesn't already tell us.

In [ ]:
stock_means = train.groupby("stock")["log_gk_volatility"].mean()
global_mean = train["log_gk_volatility"].mean()

for name, split in [
    ("val", val),
    ("test_normal", test_normal),
    ("test_covid", test_covid),
]:
    preds = split["stock"].map(stock_means).fillna(global_mean).values
    results[("PerStock", name)] = regression_metrics(
        preds, split["log_gk_volatility"].values
    )
    m = results[("PerStock", name)]
    print(f"  {name:<12}: rmse={m['rmse']:.4f}  mae={m['mae']:.4f}  r2={m['r2']:+.4f}")

unseen = split["stock"].isin(stock_means.index) == False  # noqa: E712
print(f"\ntickers in test_covid never seen in train: {unseen.sum():,} / {len(split):,}")

# Part 4: Training Infrastructure

We factor training into three reusable utilities so the three BERT variants (frozen, partial, full) share the same loop and checkpoint behavior, and differ only in what they pass in. The design mirrors the HW4 notebook: history is pickled after every epoch so a Colab disconnect never wastes a completed epoch, and a variant that's already been trained through `epochs` resumes as a no-op.

## 4.1 History I/O and Evaluation

- `load_history` / `save_history` pickle a dict of per-epoch metric lists (`train_loss`, `val_loss`, `val_rmse`, `val_mae`, `val_r2`).
- `evaluate_regressor` runs a loader end-to-end, accumulating predictions and targets, and returns a metrics dict. Used both inside the training loop (each epoch on val) and in Part 7 to score the final model on both test sub-periods.

In [ ]:
import pickle
from tqdm.auto import tqdm


# Utility functions for checkpointing, history logging, and evaluation
def load_history(path: Path) -> dict:
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return {}


def save_history(history: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(history, f)


# evaluation helper for any regressor model + regression metrics calculation
def evaluate_regressor(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    criterion: nn.Module,
) -> dict:
    model.eval()
    total_loss = 0.0
    preds_all, targets_all = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            target = batch["target"].to(device)

            pred = model(input_ids, attention_mask)
            loss = criterion(pred, target)

            total_loss += loss.item()
            preds_all.append(pred.cpu().numpy())
            targets_all.append(target.cpu().numpy())

    preds = np.concatenate(preds_all)
    targets = np.concatenate(targets_all)
    return {
        "loss": total_loss / len(loader),
        **regression_metrics(preds, targets),
    }

## 4.2 Training Loop with Checkpointing and Early Stopping

`train_regressor` handles one full training run. It uses AdamW, an optional linear-warmup LR scheduler, and gradient clipping at 1.0 (standard for BERT fine-tuning to prevent gradient explosions through 12 layers of self-attention). Behavior:

- **Resume**: if the checkpoint and history files exist, load weights and pick up at the next epoch.
- **Checkpoint**: save `state_dict()` to `checkpoint_path` whenever val loss improves.
- **Early stop**: if val loss fails to improve for `patience` consecutive epochs, stop and load the best checkpoint.
- **History**: pickle after every epoch so a crash loses at most the in-progress epoch.

The function returns the model (with best-val weights loaded) and the history dict.

In [ ]:
import copy


def train_regressor(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    checkpoint_path: Path,
    history_path: Path,
    scheduler=None,
    epochs: int = 3,
    patience: int = 2,
    grad_clip: float = 1.0,
    criterion: nn.Module = nn.MSELoss(),
) -> tuple[nn.Module, dict]:
    history = load_history(history_path)
    for k in ["train_loss", "val_loss", "val_rmse", "val_mae", "val_r2"]:
        history.setdefault(k, [])

    # inconsistency history and checkpoint will start fresh
    if history["train_loss"] and not checkpoint_path.exists():
        print(
            f"warning: history has {len(history['train_loss'])} epochs but no checkpoint; starting fresh"
        )
        history = {k: [] for k in history}
        save_history(history, history_path)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_state_dict: dict | None = None

    # Load from checkpoint if it exists
    if checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        epochs_no_improve = ckpt.get("epochs_no_improve", 0)
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        best_state_dict = ckpt.get("best_model_state_dict", None)
        print(
            f"resumed from {checkpoint_path.name}  "
            f"epoch={len(history['train_loss'])}  "
            f"no_improve={epochs_no_improve}/{patience}  "
            f"best_val_loss={best_val_loss:.4f}"
        )
    start_epoch = len(history["train_loss"])
    # if already finished
    if start_epoch >= epochs:
        print(f"already trained through epoch {start_epoch}/{epochs}")
        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)
        return model, history
    # if exceed patience
    if epochs_no_improve >= patience:
        print(f"already stopped early at epoch {start_epoch} (patience reached)")
        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)
        return model, history

    # resume training
    for epoch in range(start_epoch, epochs):
        model.train()
        if not any(p.requires_grad for p in model.bert.parameters()):
            model.bert.eval()
        total_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
        for batch in pbar:
            # only move to device inside the loop to save memory
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            target = batch["target"].to(device)

            optimizer.zero_grad()
            pred = model(input_ids, attention_mask)
            loss = criterion(pred, target)
            loss.backward()
            # gradient clipping
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()
            if scheduler is not None and not isinstance(
                scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau
            ):
                scheduler.step()

            total_loss += loss.item()
            pbar.set_postfix(
                loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}"
            )

        train_loss = total_loss / len(train_loader)
        val_metrics = evaluate_regressor(model, val_loader, device, criterion)

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_metrics["loss"])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_mae"].append(val_metrics["mae"])
        history["val_r2"].append(val_metrics["r2"])

        improved = val_metrics["loss"] < best_val_loss
        if improved:
            best_val_loss = val_metrics["loss"]
            epochs_no_improve = 0
            best_state_dict = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1

        tag = (
            "new best"
            if improved
            else f"no improvement ({epochs_no_improve}/{patience})"
        )
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
            f"val_rmse={val_metrics['rmse']:.4f} | val_r2={val_metrics['r2']:+.4f} | {tag}"
        )

        # Save full training state after every epoch
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict()
                if scheduler is not None
                else None,
                "epochs_no_improve": epochs_no_improve,
                "best_val_loss": best_val_loss,
                "best_model_state_dict": best_state_dict,
            },
            checkpoint_path,
        )
        save_history(history, history_path)

        if epochs_no_improve >= patience:
            print(f"early stopping at epoch {epoch + 1}")
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, history

## 4.3 History Plot Helpers

Two quick plotting utilities. `plot_history` visualizes a single run — train vs val loss on one panel, then val RMSE / MAE / R² on separate panels so you can spot overfitting (train↓ while val↑), underfitting (both flat), or the epoch where early stopping fired. `plot_histories` overlays multiple runs on the same axes for side-by-side variant comparison; we'll use it in the final comparison section to show how the three fine-tune depths differ epoch by epoch.

In [ ]:
def plot_history(history: dict, title: str = "") -> None:
    if not history.get("train_loss"):
        print("empty history")
        return
    epochs = list(range(1, len(history["train_loss"]) + 1))

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    axes[0].plot(epochs, history["train_loss"], "o-", label="train")
    axes[0].plot(epochs, history["val_loss"], "o-", label="val")
    axes[0].set_title("loss (MSE)")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["val_rmse"], "o-", color="steelblue")
    axes[1].set_title("val RMSE")
    axes[1].set_xlabel("epoch")

    axes[2].plot(epochs, history["val_mae"], "o-", color="steelblue")
    axes[2].set_title("val MAE")
    axes[2].set_xlabel("epoch")

    axes[3].plot(epochs, history["val_r2"], "o-", color="steelblue")
    axes[3].axhline(0, color="gray", linestyle="--", linewidth=0.8)
    axes[3].set_title("val R²")
    axes[3].set_xlabel("epoch")

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_histories(histories: dict[str, dict], title: str = "") -> None:
    """Overlay multiple runs. `histories` = {run_name: history_dict}."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    metric_specs = [
        ("val_loss", "val loss (MSE)"),
        ("val_rmse", "val RMSE"),
        ("val_mae", "val MAE"),
        ("val_r2", "val R²"),
    ]
    colors = plt.cm.tab10.colors
    for ax, (key, title_str) in zip(axes, metric_specs):
        for i, (name, h) in enumerate(histories.items()):
            if not h.get(key):
                continue
            epochs = list(range(1, len(h[key]) + 1))
            ax.plot(epochs, h[key], "o-", label=name, color=colors[i % 10])
        ax.set_title(title_str)
        ax.set_xlabel("epoch")
        if key == "val_r2":
            ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    axes[0].legend()
    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

# Part 5: Variant A — Frozen BERT (Linear Probe)

This approach freezes all BERT parameters and trains only the 769-parameter regression head. It measures how much volatility signal is already present in off-the-shelf BERT embeddings without any task-specific adaptation. Because we're essentially fitting a linear regression on top of fixed 768-dim features, we use a much higher learning rate (`1e-3` vs `2e-5`) and allow more epochs.

## 5.1 Build Model & Freeze Encoder

In [ ]:
RUN = "bert_baseline_frozen"
EPOCHS_FROZEN = 5
LR_FROZEN = 1e-3

set_seed(SEED)

model_frozen = BertRegressor().to(DEVICE)
freeze_bert_layers(model_frozen, n_unfrozen=0)
param_summary(model_frozen)

## 5.2 Train

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

trainable = [p for p in model_frozen.parameters() if p.requires_grad]
optimizer_frozen = AdamW(trainable, lr=LR_FROZEN, weight_decay=0.01)
scheduler_frozen = ReduceLROnPlateau(
    optimizer_frozen,
    mode="min",  # watching val_loss (lower is better)
    factor=0.5,  # halve the LR on plateau
    patience=0,
    min_lr=1e-8,
)

model_frozen, history_frozen = train_regressor(
    model=model_frozen,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_frozen,
    scheduler=scheduler_frozen,
    device=DEVICE,
    checkpoint_path=DATA_DIR / f"{RUN}.pt",
    history_path=DATA_DIR / f"{RUN}_history.pkl",
    epochs=EPOCHS_FROZEN,
    patience=2,
)

## 5.3 Evaluate on Test Sub-Periods

With the best-val weights loaded back into the model (handled by `train_regressor`), we score on validation, `test_normal`, and `test_covid` separately. We log the metrics into the shared `results` dict so Part 7 can compile the comparison table across all variants and baselines.

In [ ]:
criterion = nn.MSELoss()
for name, loader in [
    ("val", val_loader),
    ("test_normal", test_normal_loader),
    ("test_covid", test_covid_loader),
]:
    metrics = evaluate_regressor(model_frozen, loader, DEVICE, criterion)
    results[("Frozen", name)] = {k: metrics[k] for k in ["rmse", "mae", "r2"]}
    m = results[("Frozen", name)]
    print(f"  {name:<12}: rmse={m['rmse']:.4f}  mae={m['mae']:.4f}  r2={m['r2']:+.4f}")

In [ ]:
plot_history(history_frozen, title="Variant A — Frozen (linear probe)")

# Part 6: Variant B — Unfreeze Last 1 Layer

In [ ]:
RUN_B = "bert_baseline_unfreeze_1"
EPOCHS_B = 3
LR_B = 2e-5

set_seed(SEED)

model_b = BertRegressor().to(DEVICE)
freeze_bert_layers(model_b, n_unfrozen=1)
param_summary(model_b)

## 6.2 Train

In [ ]:
trainable = [p for p in model_b.parameters() if p.requires_grad]
optimizer_b = AdamW(trainable, lr=LR_B, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS_B
warmup_steps = int(0.1 * total_steps)
scheduler_b = get_linear_schedule_with_warmup(
    optimizer_b,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"total steps = {total_steps:,}  warmup = {warmup_steps:,}")

model_b, history_b = train_regressor(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_b,
    scheduler=scheduler_b,
    device=DEVICE,
    checkpoint_path=DATA_DIR / f"{RUN_B}.pt",
    history_path=DATA_DIR / f"{RUN_B}_history.pkl",
    epochs=EPOCHS_B,
    patience=2,
)

## 6.3 Evaluate on Test Sub-Periods

In [ ]:
criterion = nn.MSELoss()
for name, loader in [
    ("val", val_loader),
    ("test_normal", test_normal_loader),
    ("test_covid", test_covid_loader),
]:
    metrics = evaluate_regressor(model_b, loader, DEVICE, criterion)
    results[("Unfreeze-1", name)] = {k: metrics[k] for k in ["rmse", "mae", "r2"]}
    m = results[("Unfreeze-1", name)]
    print(f"  {name:<12}: rmse={m['rmse']:.4f}  mae={m['mae']:.4f}  r2={m['r2']:+.4f}")

In [ ]:
plot_history(history_b, title="Variant B — Unfreeze last 1 layer")

# Part 7: Variant C — Unfreeze Last 6 Layers

Now we unfreeze the top half of BERT (layers 6–11) while keeping layers 0–5 and embeddings frozen. This gives roughly 43M trainable parameters.

In [ ]:
RUN_C = "bert_baseline_unfreeze_6"
EPOCHS_C = 3
LR_C = 2e-5

set_seed(SEED)

model_c = BertRegressor().to(DEVICE)
freeze_bert_layers(model_c, n_unfrozen=6)
param_summary(model_c)

In [ ]:
# delete and cleanup all preious optimizers/schedulers to free up memory before creating new ones
del optimizer_b, scheduler_b
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
trainable = [p for p in model_c.parameters() if p.requires_grad]
optimizer_c = AdamW(trainable, lr=LR_C, weight_decay=0.01)

## 7.2 Train

Same training recipe as Variant B.

In [ ]:
trainable = [p for p in model_c.parameters() if p.requires_grad]
optimizer_c = AdamW(trainable, lr=LR_C, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS_C
warmup_steps = int(0.1 * total_steps)
scheduler_c = get_linear_schedule_with_warmup(
    optimizer_c,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"total steps = {total_steps:,}  warmup = {warmup_steps:,}")

model_c, history_c = train_regressor(
    model=model_c,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_c,
    scheduler=scheduler_c,
    device=DEVICE,
    checkpoint_path=DATA_DIR / f"{RUN_C}.pt",
    history_path=DATA_DIR / f"{RUN_C}_history.pkl",
    epochs=EPOCHS_C,
    patience=5,
)